<a href="https://colab.research.google.com/github/HC-AIGroup/CS5330_F26/blob/main/MODULE%207/LK_TWO_OBJECTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# ---------- Step 1: a road with TWO cars at different speeds ----------
W, H = 480, 250
def make_frame(t):
    im = np.full((H, W, 3), (200, 225, 235), np.uint8)                   # sky
    cv2.rectangle(im, (0, 120), (W, H), (95, 95, 95), -1)               # road
    for x in range(0, W, 80):                                           # lane dashes
        cv2.rectangle(im, (x, 180), (x + 40, 185), (255, 255, 255), -1)
    cv2.rectangle(im, (20, 85), (60, 120), (150, 90, 60), -1)           # house
    cv2.rectangle(im, (390, 60), (396, 120), (80, 60, 40), -1)          # sign pole
    cv2.rectangle(im, (370, 40), (416, 62), (30, 120, 40), -1)          # sign
    # red car: 6 px/frame to the RIGHT (upper lane)
    x, y = 130 + 6 * t, 130
    cv2.rectangle(im, (x, y), (x + 110, y + 35), (30, 60, 200), -1)
    cv2.rectangle(im, (x + 22, y - 22), (x + 82, y), (30, 60, 200), -1)
    cv2.rectangle(im, (x + 28, y - 17), (x + 50, y - 2), (230, 240, 250), -1)
    cv2.rectangle(im, (x + 55, y - 17), (x + 77, y - 2), (230, 240, 250), -1)
    for wx in (x + 22, x + 88):
        cv2.circle(im, (wx, y + 35), 11, (20, 20, 20), -1); cv2.circle(im, (wx, y + 35), 4, (170, 170, 170), -1)
    # green car: 3 px/frame to the LEFT (lower lane)
    x, y = 380 - 3 * t, 195
    cv2.rectangle(im, (x, y), (x + 90, y + 28), (40, 140, 40), -1)
    cv2.rectangle(im, (x + 18, y - 18), (x + 66, y), (40, 140, 40), -1)
    cv2.rectangle(im, (x + 23, y - 14), (x + 40, y - 2), (230, 240, 250), -1)
    cv2.rectangle(im, (x + 45, y - 14), (x + 62, y - 2), (230, 240, 250), -1)
    for wx in (x + 18, x + 72):
        cv2.circle(im, (wx, y + 28), 9, (20, 20, 20), -1); cv2.circle(im, (wx, y + 28), 3, (170, 170, 170), -1)
    return im

frame1, frame2 = make_frame(0), make_frame(1)
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

# ---------- Step 2: corners in frame 1 (Shi-Tomasi) ----------
p1 = cv2.goodFeaturesToTrack(gray1, maxCorners=80, qualityLevel=0.05,
                             minDistance=8, blockSize=3)
print("corners found:", len(p1))

# ---------- Step 3: Lucas-Kanade ----------
p2, status, err = cv2.calcOpticalFlowPyrLK(gray1, gray2, p1, None,
                                           winSize=(15, 15), maxLevel=2)
ok = status.ravel() == 1
old, new = p1[ok].reshape(-1, 2), p2[ok].reshape(-1, 2)

# ---------- Step 4: motion vector per corner ----------
motion = new - old                                  # (u, v) for each corner

# ---------- Step 5: moving or still? ----------
moving = np.linalg.norm(motion, axis=1) > 1.0
print("moving corners:", moving.sum(), " still corners:", (~moving).sum())

# ---------- Step 6 (NEW): group moving corners by similar motion ----------
# points whose (u, v) differ by less than 1 px end up in the same group = same object
labels = DBSCAN(eps=1.0, min_samples=3).fit_predict(motion[moving])
mpts, mmot = old[moving], motion[moving]
npts = new[moving]                                  # positions in frame 2
for k in sorted(set(labels) - {-1}):
    g = labels == k
    u, v = mmot[g].mean(axis=0)
    x1, y1 = mpts[g].min(axis=0); x2, y2 = mpts[g].max(axis=0)
    print(f"object {k+1}: {g.sum()} corners, motion (u, v) = ({u:.2f}, {v:.2f}) px/frame, "
          f"box x {x1:.0f}-{x2:.0f}, y {y1:.0f}-{y2:.0f}")

# ---------- Step 7: draw ----------
colors = ["red", "blue", "orange", "purple"]
plt.figure(figsize=(9, 4.7))
plt.imshow(cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB))
for (x0, y0) in old[~moving]:
    plt.plot(x0, y0, "o", color="yellow", markeredgecolor="black")
for k in sorted(set(labels) - {-1}):
    g = labels == k; c = colors[k % 4]
    for (x0, y0), (u, v) in zip(mpts[g], mmot[g]):
        plt.arrow(x0, y0, 5 * u, 5 * v, color=c, width=1.2, head_width=5)
    x1, y1 = npts[g].min(axis=0) - 6; x2, y2 = npts[g].max(axis=0) + 6   # box around the group in frame 2
    plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=c, lw=2, ls="--"))
    plt.text(x1, y1 - 4, f"object {k+1}: u = {mmot[g][:,0].mean():.1f}", color=c, fontsize=11, weight="bold")
plt.title("each group = one object (arrows x5), yellow = still")
plt.axis("off")
plt.show()

# ---------- Step 8 (optional): speed of each car ----------
FPS, PX_PER_M = 30, 110 / 4.5
for k in sorted(set(labels) - {-1}):
    u = mmot[labels == k][:, 0].mean()
    print(f"object {k+1}: {abs(u)*FPS/PX_PER_M*3.6:.0f} km/h, moving {'right' if u > 0 else 'left'}")